# sub-16: recovering the bars sequences — validation, combined evidence, self-training

**Where we are** (`sub16_stimulus_decoder.ipynb`):
- For BarsLeftEye run-01, the decoder and the free pRF search give **exactly the same order**.
- RE01 and RE02 remain uncertain (2-3 sweeps).
- A decoder trained on rings/wedges only reads bars badly (5/9 on LE02).

**This notebook**

1. **Validation in the right configuration**: decoding a run whose sequence the decoder has never seen.
   - **V1**: sub-16 LE02 (A), with a decoder trained on rings/wedges only.
   - **V2**: sub-05 RightEye Bars01, which is **B**, with a decoder trained on rings/wedges + its 3 **A** runs. This is exactly the configuration of the suspect runs (bars of another sequence in training).
   - **V3, V4**: sub-13 (A) and sub-17 (B), rings/wedges only.

   The sweeps found are counted, and the **margins** of correct and wrong sweeps give a reliability threshold.
2. **Combined evidence**: decoded movie + pRF model, each normalised, summed frame by frame. Exact search (dynamic programming):
   - free;
   - with the structure of the A generator (8 directions once each, one inserted half blank).

   Each sweep gets a global margin (best chain minus the best chain with another direction at this place).
3. **Self-training**: the decoder is retrained adding the other suspect runs with their recovered chain (passes 1 and 2), and the stability of the chains is checked.
4. **Round trip** (the key test): a decoder trained on rings/wedges + the 3 suspect runs **with their recovered chains**, never seeing LE02, decodes LE02.
   - If the recovered chains are right, it must find A better than with rings/wedges alone.
   - It must also do better than with the same timings but **random directions** (control).

Read only. Design matrices in `Desktop/fig_od/sub16_combined/`.

In [1]:
import os, glob, math
from functools import lru_cache
import numpy as np, pandas as pd, nibabel as nb
import scipy.io as sio
import matplotlib.pyplot as plt

main = '/Users/uriel/disks/meso_shared/amblyo7T_prf'
pp = f'{main}/derivatives/pp_data'
out_dir = '/Users/uriel/Desktop/fig_od/sub16_combined'
os.makedirs(out_dir, exist_ok=True)
TR = 2.0
lags = [1, 2, 3, 4]                       # decoder: data at t+lag decodes the stimulus at t
model_lags = [2, 3]                       # pRF evidence: response pattern at t+lag
alphas = [1e1, 1e2, 1e3, 1e4, 1e5, 1e6]
dec_rois, dec_nmax = [1, 2, 3], 3000      # best variant of sub16_stimulus_decoder.ipynb
RW = ['RingsLeftEye_run-01', 'WedgesLeftEye_run-01', 'RingsRightEye_run-01', 'WedgesRightEye_run-01']
SUSPECTS = ['BarsLeftEye_run-01', 'BarsRightEye_run-01', 'BarsRightEye_run-02']
rng = np.random.default_rng(0)


/var/folders/60/q71xm4hx45x3db5d6mzl8r4h0000gn/T/ipykernel_6256/3008775802.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import numpy as np, pandas as pd, nibabel as nb


## Functions (stimuli, data, decoder, pRF model — same as the decoder notebook)

In [2]:
def build_vdm(mat_fn):
    """Same conversion as create_vdm_files.ipynb (50 x 50, transposed, padded)."""
    im = np.asarray(sio.loadmat(mat_fn, squeeze_me=True, struct_as_record=False)['s'].im, float)
    out = np.zeros((50, 50, im.shape[-1]))
    ii = np.floor(np.arange(50) * 102 / 50).astype(int)
    for t in range(im.shape[-1]):
        f = (im[..., t] * 255).astype(np.uint8).T
        sq = np.vstack([np.zeros((12, 102), np.uint8), f, np.zeros((13, 102), np.uint8)])
        out[..., t] = (sq > 5)[np.ix_(ii, ii)]
    return out

vdm = {eye: np.load(f'{main}/derivatives/vdm/task-BarsBarsRingsWedges{eye}_vdm.npy') for eye in ['LeftEye', 'RightEye']}
A = vdm['LeftEye'][..., :120]
B = build_vdm(f'{main}/sourcedata/C/jp/cond-1-stim-3.mat')
true_stim = {'A': A, 'B': B}


def run_stim(r, kind=None):
    """Known stimulus (50,50,120) of a run."""
    eye = 'LeftEye' if 'LeftEye' in r else 'RightEye'
    if r.startswith('Rings'): return vdm[eye][..., 240:360]
    if r.startswith('Wedges'): return vdm[eye][..., 360:480]
    return true_stim[kind]


def ds(st):
    """(50,50,T) -> (T, 625), 2x2 average."""
    return st.reshape(25, 2, 25, 2, -1).mean((1, 3)).reshape(625, -1).T

rr, cc = np.meshgrid(np.arange(50) - 24.5, np.arange(50) - 24.5, indexing='ij')
X_pix, Y_pix = cc, -rr


def frame_label(f):
    m = f > 0
    if m.sum() < 3: return None, np.nan
    best = None
    for a in [0, 45, 90, 135]:
        th = np.deg2rad(a); p = X_pix[m] * np.cos(th) + Y_pix[m] * np.sin(th)
        if best is None or p.std() < best[0]: best = (p.std(), a, p.mean())
    return best[1], best[2]


def segment(stim):
    lab = [frame_label(stim[..., t]) for t in range(stim.shape[-1])]
    ev, t = [], 0
    while t < len(lab):
        o, p = lab[t]; u = t + 1
        if o is None:
            while u < len(lab) and lab[u][0] is None: u += 1
            ev.append(('b', t, u - t))
        else:
            while u < len(lab) and lab[u][0] == o and u - t < 12: u += 1
            ev.append((f"{o}{'+' if lab[u - 1][1] > p else '-'}", t, u - t))
        t = u
    return ev

templates = {}
for stim in [A, B]:
    for k, s, l in segment(stim):
        if k != 'b' and l == 12 and k not in templates:
            templates[k] = stim[..., s:s + 12]
print(sorted(templates))


def tmpl(k, L):
    """Sweep k resampled to L frames, full resolution (50,50,L)."""
    return templates[k][..., np.round(np.linspace(0, 11, L)).astype(int)]


def chain_to_stim(chain):
    st = np.zeros((50, 50, 120))
    for k, t0, L in chain:
        if k == 'b': continue
        fr = tmpl(k, L)
        n = min(L, 120 - t0)
        st[..., t0:t0 + n] = fr[..., :n]
    return st


def chain_str(chain):
    return ' '.join(f'b{L}' if k == 'b' else (k if L == 12 else f'{k}/{L}') for k, t0, L in chain)

true_chain = {k: [(e[0], e[1], e[2]) for e in segment(v)] for k, v in true_stim.items()}
print('A:', chain_str(true_chain['A'])); print('B:', chain_str(true_chain['B']))

['0+', '0-', '135+', '135-', '45+', '45-', '90+', '90-']
A: b6 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135-/1
B: 0+ 135- 90- 45- 0- 135+ 90+ 45+ 0+ 135-


In [3]:
cache = {}
def load_subject(s):
    if s in cache: return cache[s]
    roi = np.concatenate([nb.load(glob.glob(f'{pp}/{s}/fsnative/rois/{s}_hemi-{h}_fmriprep_dct_*_rois-group-mmp.func.gii')[0]).darrays[0].data
                          for h in ['L', 'R']])
    prf = {eye: np.hstack([np.vstack([a.data for a in nb.load(
        f'{pp}/{s}/fsnative/prfmodel/prf_derivatives/{s}_task-RingsWedges{eye}_hemi-{h}_fmriprep_dct_psc_concat_prf-css_deriv.func.gii').darrays])
        for h in ['L', 'R']]) for eye in ['LeftEye', 'RightEye']}
    cache[s] = (roi, prf)
    return cache[s]

run_cache = {}
def load_run(s, r):
    if (s, r) in run_cache: return run_cache[(s, r)]
    hs = []
    for h in ['L', 'R']:
        d = np.vstack([a.data for a in nb.load(
            f'{pp}/{s}/fsnative/func/fmriprep_dct_psc/{s}_ses-01_task-{r}_hemi-{h}_space-fsnative_fmriprep_dct_psc_bold.func.gii').darrays])
        hs.append(d if d.shape[0] == 120 else d.T)
    run_cache[(s, r)] = np.hstack(hs).astype(np.float32)
    return run_cache[(s, r)]


def select_vertices(s, rois, n_max):
    roi, prf = load_subject(s)
    r2 = (prf['LeftEye'][0] + prf['RightEye'][0]) / 2
    ok = np.isfinite(r2) & (r2 > 0.1) & ((roi > 0) if rois is None else np.isin(roi, rois))
    idx = np.where(ok)[0]
    return idx[np.argsort(-r2[idx])[:n_max]]


def zc(a, axis=0):
    a = a - a.mean(axis, keepdims=True)
    return a / (a.std(axis, keepdims=True) + 1e-9)


def features(y):
    """(120, nv) -> (120, nv * n_lags): row t = data at t+lag."""
    y = zc(np.nan_to_num(y))
    cols = []
    for l in lags:
        z = np.zeros_like(y); z[:120 - l] = y[l:]
        cols.append(z)
    return np.hstack(cols) / np.sqrt(y.shape[1] * len(lags))

In [4]:
def frame_corr(D, S):
    """Median over non-blank frames of the correlation between decoded and true frames."""
    m = S.sum(1) > 0
    Dc = D[m] - D[m].mean(1, keepdims=True); Sc = S[m] - S[m].mean(1, keepdims=True)
    return float(np.median((Dc * Sc).sum(1) / (np.sqrt((Dc ** 2).sum(1) * (Sc ** 2).sum(1)) + 1e-9)))


def fit_predict(Xtr, Ytr, Xte, alpha):
    mu = Ytr.mean(0)
    K = Xtr @ Xtr.T
    W = np.linalg.solve(K + alpha * np.eye(len(K)), Ytr - mu)
    return Xte @ Xtr.T @ W + mu


def train_decoder(train):
    """train: list of (X, Y). Returns predict(X), chosen alpha, gain, LORO frame corr."""
    best = None
    for a in alphas:
        preds = [fit_predict(np.vstack([t[0] for j, t in enumerate(train) if j != i]),
                             np.vstack([t[1] for j, t in enumerate(train) if j != i]), train[i][0], a) for i in range(len(train))]
        sc = np.mean([frame_corr(p, t[1]) for p, t in zip(preds, train)])
        if best is None or sc > best[0]:
            best = (sc, a, preds)
    sc, a, preds = best
    P, Y = np.vstack(preds), np.vstack([t[1] for t in train])
    mu = Y.mean(0)
    gain = float(((P - mu) * (Y - mu)).sum() / ((Y - mu) ** 2).sum())   # shrinkage of the decoded amplitude
    Xtr, Ytr = np.vstack([t[0] for t in train]), np.vstack([t[1] for t in train])
    return (lambda X: fit_predict(Xtr, Ytr, X, a)), a, gain, sc, mu

In [5]:
pix = 2 * math.degrees(math.atan(50.6 / 2 / 242)) / 50
Xd, Yd = X_pix * pix, Y_pix * pix
t_ = np.arange(0, 32, TR)
g = lambda a: t_ ** (a - 1) * np.exp(-t_) / math.gamma(a)
H = g(6) - g(16) / 6; H /= H.sum()
N_fft = 256
Hf = np.fft.rfft(H, n=N_fft)


def conv_hrf(d):
    return np.fft.irfft(np.fft.rfft(d, n=N_fft, axis=0) * Hf[:, None], n=N_fft, axis=0)[:d.shape[0]]


def prf_score(stim, s, r):
    """Median correlation (V1-V3 vertices, pRFs of the eye) between data and pRF prediction, best delay -2..+2 TR."""
    roi, prf = load_subject(s)
    D = prf['LeftEye' if 'LeftEye' in r else 'RightEye']
    y = load_run(s, r)
    idx = np.where((D[0] > 0.2) & (D[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
    x0, y0, sd, n = D[7][idx], D[8][idx], np.clip(D[4][idx], 0.2, None), np.clip(D[11][idx], 0.05, 1)
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
    p = conv_hrf(np.clip(np.moveaxis(stim, -1, 0).reshape(120, -1) @ G, 0, None) ** n)
    yy = y[:, idx]
    best = -np.inf
    for sh in range(-2, 3):
        q = np.roll(p, sh, 0)
        if sh > 0: q[:sh] = p[0]
        elif sh < 0: q[sh:] = p[-1]
        best = max(best, float(np.median((zc(q) * zc(yy)).mean(0))))
    return best

## Frame evidence and exact searches

In [6]:
keys = sorted(templates)
T = 120


def decoder_scores(D, gain, mu):
    """Frame scores from the decoded movie, relative to a blank frame: dict k -> (T, 12)."""
    Dc = D - mu
    blank_err = ((Dc + gain * mu) ** 2).sum(1)
    out = {}
    for k in keys:
        F = ds(tmpl(k, 12))
        err = ((Dc[:, None, :] - gain * (F - mu)[None]) ** 2).sum(2)
        out[k] = blank_err[:, None] - err
    return out


model_cache = {}
def model_scores(s, r):
    """Frame scores from the pRF model: spatial correlation between the response pattern at t+lag and the pRF drive of each template frame."""
    if (s, r) in model_cache: return model_cache[(s, r)]
    roi, prf = load_subject(s)
    Dp = prf['LeftEye' if 'LeftEye' in r else 'RightEye']
    y = load_run(s, r)
    idx = np.where((Dp[0] > 0.2) & (Dp[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
    x0, y0, sd, n = Dp[7][idx], Dp[8][idx], np.clip(Dp[4][idx], 0.2, None), np.clip(Dp[11][idx], 0.05, 1)
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
    Y = zc(y[:, idx])
    out = {}
    for k in keys:
        drv = np.clip(np.moveaxis(tmpl(k, 12), -1, 0).reshape(12, -1) @ G, 0, None) ** n
        Dz = zc(drv, axis=1)
        acc = 0
        for l in model_lags:
            Yl = np.zeros_like(Y); Yl[:T - l] = Y[l:]
            acc = acc + zc(Yl, axis=1) @ Dz.T / len(idx)
        out[k] = acc / len(model_lags)
    model_cache[(s, r)] = out
    return out


def normalise(S, center):
    allv = np.concatenate([S[k].ravel() for k in keys])
    c = np.median(allv) if center else 0.0
    sd = allv.std() + 1e-12
    return {k: (S[k] - c) / sd for k in keys}


def combine(*Ss):
    return {k: sum(S[k] for S in Ss) for k in keys}


def sweep_sums(S):
    """Score of a 12-TR sweep of k starting at t0 (cut at the end of the run)."""
    return {k: np.array([sum(S[k][t0 + i, i] for i in range(12) if t0 + i < T) for t0 in range(T)]) for k in keys}


In [7]:
def merge_blanks(chain):
    out = []
    for e in chain:
        if out and e[0] == 'b' and out[-1][0] == 'b':
            out[-1] = ('b', out[-1][1], out[-1][2] + e[2])
        else:
            out.append(e)
    return out


def dp_free(S):
    """Exact best chain, no constraint: sweeps (8 directions, 12 TR) and blanks (1-14 TR, score 0).
    Returns chain [(k, t0, L)], total score, and for each sweep the global margin over the best chain with another
    direction at the same place (max-marginals)."""
    sc = sweep_sums(S)
    fwd = np.full(T + 1, -np.inf); fwd[0] = 0; back = [None] * (T + 1)
    for t in range(T):
        if not np.isfinite(fwd[t]): continue
        for b in range(1, 15):
            u = min(T, t + b)
            if fwd[t] > fwd[u]: fwd[u] = fwd[t]; back[u] = (t, 'b', u - t)
        for k in keys:
            u = min(T, t + 12); v = fwd[t] + sc[k][t]
            if v > fwd[u]: fwd[u] = v; back[u] = (t, k, 12)
    bwd = np.full(T + 1, -np.inf); bwd[T] = 0
    for t in range(T - 1, -1, -1):
        bwd[t] = max(max(bwd[min(T, t + b)] for b in range(1, 15)), max(sc[k][t] + bwd[min(T, t + 12)] for k in keys))
    chain, u = [], T
    while u > 0:
        t, k, l = back[u]; chain.append((k, t, l)); u = t
    chain = merge_blanks(chain[::-1])
    opt = fwd[T]
    conf = []
    for k, t0, l in chain:
        if k == 'b': continue
        alt = max(fwd[t0] + sc[k2][t0] + bwd[min(T, t0 + 12)] for k2 in keys if k2 != k)
        conf.append((k, t0, opt - alt))
    return chain, opt, conf


def dp_perm(S, init_blank=range(3, 8), ins_blank=(5, 6)):
    """Exact best chain with the generator structure seen in A: initial blank, the 8 directions once each with one
    inserted half-cycle blank, then free sweeps until the end (at least 7 directions completed)."""
    sc = sweep_sums(S)
    full = (1 << len(keys)) - 1

    @lru_cache(maxsize=None)
    def V(t, mask, bl, pb):
        if t >= T:
            return (0.0 if bin(mask).count('1') >= 7 else -np.inf), None
        best = (-np.inf, None)
        for i, k in enumerate(keys):
            if mask == full or not (mask >> i) & 1:
                nm = mask if mask == full else mask | (1 << i)
                v = sc[k][t] + V(min(T, t + 12), nm, bl, False)[0]
                if v > best[0]: best = (v, (k, 12, nm, bl, False))
        if mask != full and not bl and mask and not pb:
            for b in ins_blank:
                v = V(min(T, t + b), mask, True, True)[0]
                if v > best[0]: best = (v, ('b', b, mask, True, True))
        return best

    starts = [(V(l, 0, False, True)[0], l) for l in init_blank]
    opt, l0 = max(starts)
    chain, t, state = [('b', 0, l0)], l0, (0, False, True)
    while t < T:
        _, ch = V(t, *state)
        k, l, nm, bl, pb = ch
        chain.append((k, t, min(l, T - t) if k == 'b' else 12))
        t, state = t + l, (nm, bl, pb)
    V.cache_clear()
    return chain, opt


def sweeps_of(chain):
    return [(k, t0) for k, t0, l in chain if k != 'b' and t0 + 12 <= T]


def n_correct(chain, true):
    """Number of full sweeps of the true chain found in the chain (same direction, start +-1 TR) / number of true sweeps."""
    est = sweeps_of(chain)
    tr = sweeps_of(true)
    ok = sum(any(k == k2 and abs(t - t2) <= 1 for k2, t2 in est) for k, t in tr)
    return f'{ok}/{len(tr)}'


def agreement(c1, c2):
    a, b = sweeps_of(c1), sweeps_of(c2)
    ok = sum(any(k == k2 and abs(t - t2) <= 1 for k2, t2 in b) for k, t in a)
    return f'{ok}/{len(a)}'


def cstr(chain):
    return ' '.join(f'b{l}' if k == 'b' else k for k, t0, l in chain)


In [8]:
calib = []   # (case, direction, start, margin, correct) for the sweeps of the validation runs
vert_cache = {}
def decode(s, r, train):
    """train: list of (run, stim (50,50,120)). Returns decoded movie, gain, mu, LORO frame corr."""
    if s not in vert_cache: vert_cache[s] = select_vertices(s, dec_rois, dec_nmax)
    idx = vert_cache[s]
    tr = [(features(load_run(s, rr)[:, idx]), ds(st)) for rr, st in train]
    pred, a, gain, loro, mu = train_decoder(tr)
    return pred(features(load_run(s, r)[:, idx])), gain, mu, loro


def analyse(s, r, train, true=None, label=''):
    D, gain, mu, loro = decode(s, r, train)
    Sd = normalise(decoder_scores(D, gain, mu), center=False)
    Sm = normalise(model_scores(s, r), center=True)
    Sc = combine(Sd, Sm)
    res = {'case': label, 'subject': s, 'run': r, 'LORO frame corr (train)': loro}
    chains = {}
    for name, S in [('decoder', Sd), ('pRF model', Sm), ('combined', Sc)]:
        ch, opt, conf = dp_free(S)
        chains[name] = ch
        res[f'{name}: chain'] = cstr(ch)
        if name == 'combined':
            res['combined: min margin'] = min(c_[2] for c_ in conf) if conf else np.nan
            res['combined: margins'] = ' '.join(f'{k}@{t0}:{m:.1f}' for k, t0, m in conf)
            res['_conf'] = conf
            if true is not None:
                tr = sweeps_of(true)
                for k, t0, m in conf:
                    if t0 + 12 <= T:
                        calib.append((label, k, t0, m, any(k == k2 and abs(t0 - t2) <= 1 for k2, t2 in tr)))
        if true is not None:
            res[f'{name}: correct'] = n_correct(ch, true)
    chp, optp = dp_perm(Sc)
    chains['combined + permutation'] = chp
    res['combined + permutation: chain'] = cstr(chp)
    res['free vs permutation (agreement)'] = agreement(chains['combined'], chp)
    if true is not None:
        res['combined + permutation: correct'] = n_correct(chp, true)
        res['frame corr decoded vs true'] = frame_corr(D, ds(chain_to_stim(true)))
    res['pRF score combined'] = prf_score(chain_to_stim(chains['combined']), s, r)
    res['pRF score combined + permutation'] = prf_score(chain_to_stim(chp), s, r)
    return res, chains, D


def show(rows, cols=None):
    df = pd.DataFrame([{k: v for k, v in r_.items() if not k.startswith('_')} for r_ in rows]).set_index('case')
    pd.set_option('display.max_colwidth', 120)
    print(df[cols].T.to_string() if cols else df.T.to_string())
    return df

stim_of = lambda r: run_stim(r)                    # known stimuli of rings / wedges


## Step 1 — validation (decoding runs whose sequence was never seen)

In [9]:
val_rows = []
cases = [
    ('V1', 'sub-16', 'BarsLeftEye_run-02', 'A', [(r, run_stim(r)) for r in RW]),
    ('V2', 'sub-05', 'BarsRightEye_run-01', 'B', [(r, run_stim(r)) for r in RW] +
                                             [(r, A) for r in ['BarsLeftEye_run-01', 'BarsLeftEye_run-02', 'BarsRightEye_run-02']]),
    ('V3', 'sub-13', 'BarsLeftEye_run-01', 'A', [(r, run_stim(r)) for r in RW]),
    ('V4', 'sub-17', 'BarsLeftEye_run-01', 'B', [(r, run_stim(r)) for r in RW]),
]
for name, s, r, kind, train in cases:
    lab = f'{name} {s} {r.replace("Eye_run-0", "")} ({kind}) | train: RW' + (' + 3 A runs' if len(train) > 4 else '')
    res, ch, _ = analyse(s, r, train, true_chain[kind], lab)
    val_rows.append(res)
    print(lab, 'done', flush=True)
print('true A:', cstr(true_chain['A'])); print('true B:', cstr(true_chain['B']))
val = show(val_rows, ['decoder: correct', 'pRF model: correct', 'combined: correct', 'combined + permutation: correct',
                      'combined: chain', 'combined + permutation: chain', 'combined: margins', 'frame corr decoded vs true',
                      'LORO frame corr (train)'])
cal = pd.DataFrame(calib, columns=['case', 'direction', 'start', 'margin', 'correct'])
print('\nMargins of the combined sweeps of the validation runs (correct vs wrong):')
print(cal.groupby('correct')['margin'].describe().round(2).to_string())
thr = cal.loc[~cal['correct'], 'margin'].max() if (~cal['correct']).any() else 0.0
print('\nReliability threshold = largest margin of a wrong sweep = {:.2f}'.format(thr))
print('Fraction of correct sweeps above it: {:.0%}'.format((cal.loc[cal['correct'], 'margin'] > thr).mean()))


V1 sub-16 BarsLeft2 (A) | train: RW done
V2 sub-05 BarsRight1 (B) | train: RW + 3 A runs done
V3 sub-13 BarsLeft1 (A) | train: RW done
V4 sub-17 BarsLeft1 (B) | train: RW done
true A: b6 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135-
true B: 0+ 135- 90- 45- 0- 135+ 90+ 45+ 0+ 135-
case                                                                                           V1 sub-16 BarsLeft2 (A) | train: RW                                                                     V2 sub-05 BarsRight1 (B) | train: RW + 3 A runs                                                                   V3 sub-13 BarsLeft1 (A) | train: RW                                                                        V4 sub-17 BarsLeft1 (B) | train: RW
decoder: correct                                                                                                               5/9                                                                                                               10/10                  

## Step 2 — suspect runs: pass 0 (training: RW + LE02) and self-training passes 1-2

In [10]:
def train_base():
    return [(r, run_stim(r)) for r in RW] + [('BarsLeftEye_run-02', A)]

passes, rows_by_pass = {}, {}
# pass 0: decoder trained on RW + LE02 (A)
rows0, ch0 = [], {}
for r in SUSPECTS:
    res, ch, _ = analyse('sub-16', r, train_base(), None, f'pass 0 | {r}')
    rows0.append(res); ch0[r] = ch
    print(r, 'done', flush=True)
passes[0] = ch0; rows_by_pass[0] = rows0
show(rows0, ['combined: chain', 'combined + permutation: chain', 'free vs permutation (agreement)', 'combined: margins',
             'decoder: chain', 'pRF model: chain', 'pRF score combined', 'pRF score combined + permutation'])

# passes 1 and 2: the two other suspect runs are added to the training set with their chain from the previous pass
for p in [1, 2]:
    rows_p, chp = [], {}
    for r in SUSPECTS:
        train = train_base() + [(o, chain_to_stim(passes[p - 1][o]['combined'])) for o in SUSPECTS if o != r]
        res, ch, _ = analyse('sub-16', r, train, None, f'pass {p} | {r}')
        res['agreement with previous pass'] = agreement(passes[p - 1][r]['combined'], ch['combined'])
        rows_p.append(res); chp[r] = ch
        print(f'pass {p}', r, 'done', flush=True)
    passes[p] = chp; rows_by_pass[p] = rows_p
    show(rows_p, ['combined: chain', 'agreement with previous pass', 'combined + permutation: chain',
                  'free vs permutation (agreement)', 'combined: margins', 'pRF score combined', 'pRF score combined + permutation'])


BarsLeftEye_run-01 done
BarsRightEye_run-01 done
BarsRightEye_run-02 done
case                                                                                                          pass 0 | BarsLeftEye_run-01                                                                            pass 0 | BarsRightEye_run-01                                                                           pass 0 | BarsRightEye_run-02
combined: chain                                                                          b5 90+ 0+ b5 45+ b1 135- 45- 135+ 90- 135- 0- b1                                                         b5 135- 0+ 135- 45- b1 135+ 90+ b4 0- 0+ 90- b2                                                           b5 135- 0- 0+ 135+ 45- b6 135- 90+ 90- 0+ b1
combined + permutation: chain                                                                b5 90+ 0+ b5 45+ 135- 0- 135+ 45- 90- 0- 45+                                                          b5 90- 0+ 135- 45- 135+ 90+ b6 0- 45+ 90-

## Step 3 — round trip: decoding LE02 from the recovered chains

In [11]:
final = {r: passes[2][r] for r in SUSPECTS}


def randomise(chain):
    """Same timing, random directions (a random permutation of the 8 directions, cycled)."""
    perm = list(rng.permutation(keys)); out, i = [], 0
    for k, t0, l in chain:
        if k == 'b': out.append((k, t0, l))
        else: out.append((perm[i % 8], t0, l)); i += 1
    return out

rt_rows = []
# baseline: RW only (same as V1)
res, _, _ = analyse('sub-16', 'BarsLeftEye_run-02', [(r, run_stim(r)) for r in RW], true_chain['A'], 'LE02 | train RW only')
rt_rows.append(res)
# recovered chains of the 3 suspect runs (LE02 never used)
for variant in ['combined', 'combined + permutation']:
    train = [(r, run_stim(r)) for r in RW] + [(o, chain_to_stim(final[o][variant])) for o in SUSPECTS]
    res, _, _ = analyse('sub-16', 'BarsLeftEye_run-02', train, true_chain['A'], f'LE02 | train RW + 3 suspects ({variant} chains)')
    rt_rows.append(res)
# control: same timings, random directions
for i in range(3):
    train = [(r, run_stim(r)) for r in RW] + [(o, chain_to_stim(randomise(final[o]['combined']))) for o in SUSPECTS]
    res, _, _ = analyse('sub-16', 'BarsLeftEye_run-02', train, true_chain['A'], f'LE02 | train RW + 3 suspects (random directions #{i + 1})')
    rt_rows.append(res)
    print('random', i + 1, 'done', flush=True)
print('true A:', cstr(true_chain['A']))
rt = show(rt_rows, ['decoder: correct', 'combined: correct', 'combined + permutation: correct', 'frame corr decoded vs true',
                    'decoder: chain', 'combined: chain'])
print('''
Round trip: if the recovered chains of the suspect runs are right, a decoder that learns from them (and never sees LE02)
must decode LE02 better than a decoder trained on rings/wedges only, and better than with random directions at the same times.''')


random 1 done
random 2 done
random 3 done
true A: b6 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135-
case                                                    LE02 | train RW only   LE02 | train RW + 3 suspects (combined chains) LE02 | train RW + 3 suspects (combined + permutation chains) LE02 | train RW + 3 suspects (random directions #1) LE02 | train RW + 3 suspects (random directions #2) LE02 | train RW + 3 suspects (random directions #3)
decoder: correct                                                         5/9                                              8/9                                                          9/9                                                 7/9                                                 4/9                                                 3/9
combined: correct                                                        7/9                                              8/9                                                          9/9                              

## Step 4 — final chains and reliability of each sweep

In [12]:
rows_f = []
for i, r in enumerate(SUSPECTS):
    ch = final[r]
    conf = rows_by_pass[2][i]['_conf']
    reliable = [f'{k}@{t0}' + ('' if m > thr else '?') for k, t0, m in conf]
    rows_f.append({'run': r,
                   'final chain (combined)': cstr(ch['combined']),
                   'final chain (combined + permutation)': cstr(ch['combined + permutation']),
                   'sweeps (? = margin below the threshold of the validation runs)': ' '.join(reliable),
                   'n reliable sweeps': f"{sum(m > thr for _, _, m in conf)}/{len(conf)}",
                   'agreement pass 0 / pass 2': agreement(passes[0][r]['combined'], ch['combined']),
                   'pRF score (combined)': rows_by_pass[2][i]['pRF score combined'],
                   'pRF score (combined + permutation)': rows_by_pass[2][i]['pRF score combined + permutation']})
    for variant in ['combined', 'combined + permutation']:
        np.save(f"{out_dir}/sub-16_{r}_{variant.replace(' + ', '_').replace(' ', '_')}_vdm.npy", chain_to_stim(ch[variant]))
fin = pd.DataFrame(rows_f).set_index('run')
print(fin.T.to_string())
print('\nDesign matrices saved in', out_dir)


run                                                                                                          BarsLeftEye_run-01                                             BarsRightEye_run-01                                             BarsRightEye_run-02
final chain (combined)                                                          b5 90+ 0+ b5 0+ b1 135- 45- 135+ 90- 135- 0- b1                 b5 135- 0+ 135- 45- 135+ 90+ b5 0- 0+ b1 90- b1                    b5 135- 0- 0+ 135+ 45- b6 135- 90+ 90- 0+ b1
final chain (combined + permutation)                                               b5 90+ b6 0- 45+ 0+ 45- 135+ 90- 135- 0- 45-                   b5 90- 0+ 135- 45- 135+ 90+ b6 0- 45+ 90- 45-                    b5 135- 0- 45+ 135+ 45- b6 0+ 90+ 90- 0+ 45-
sweeps (? = margin below the threshold of the validation runs)  90+@5 0+@17 0+@34? 135-@47 45-@59 135+@71 90-@83 135-@95 0-@107  135-@5 0+@17 135-@29 45-@41 135+@53 90+@65 0-@82 0+@94 90-@107  135-@5 0-@17 0+@29 135+@41 45-@53 135-@

## How to read

- **Step 1**: how many sweeps the method recovers on runs of unseen sequence, especially **V2** (same configuration as the suspect runs). The threshold is the largest margin among wrong sweeps: a sweep above it is reliable.
- **Step 2**: chains stable from one pass to the next, and free = permutation.
- **Step 3**: the decisive test. LE02 decoded from the recovered chains must be:
  - clearly better than with RW only (correct sweeps, frame corr);
  - clearly better than with random directions.

  Only correct chains can teach the decoder to read A.
- **Step 4**: final chain of each suspect run, with the uncertain sweeps marked `?`. If the round trip succeeds and few `?` remain, the design matrices can be used.